# Model Finetuning
We will finetune the model in the following steps:
- Take in the finetuned dataset
- Convert to the proper format (tensors)
- Then finetune the model based

Some imports for libaries we will use and helper classes for finetuning the model. Instead of writing our own training loop and evaluation function, we will be using the given functions from pytorch directly. 

In [1]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import json # For our manually labeled finetune train data
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset
import os
import sys

base = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection"
files = ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]

for f in files:
    os.system(f"curl -o finetune_helpers/{f} {base}/{f}")

sys.path.append("./finetune_helpers")
from finetune_helpers.engine import train_one_epoch
from finetune_helpers import utils

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063  100  4063    0     0  56309      0 --:--:-- --:--:-- --:--:-- 56430
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8388  100  8388    0     0   105k      0 --:--:-- --:--:-- --:--:--  106k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8409  100  8409    0     0   114k      0 --:--:-- --:--:-- --:--:--  115k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6447  100  6447    0     0  97714      0 --:--:-- --:--:-- --:--:-- 99184
  % Total    % Received % Xferd  Average Speed   Tim

# Steps 1 & 2
First we will take in the finetuned dataset (*annotations.json*) and convert it into a list of dictionaries where each element of the dictinoary is either "boxes", i.e. the location of boxes on our classified images, or "labels", i.e. 1 since we are classifying tree (1) and not tree (0). 
We do this in the __init__ function.

Since we will be working with the data when training/finetuning we create a class that inherits from Dataset. 

In [2]:
class TreeDataset(Dataset):
    def __init__(self, annotations_path):
        self.root_path = annotations_path
        with open(self.root_path + "annotations.json") as f:
            all_annotations = json.load(f)

        self.annotations = []
        for a in all_annotations:
            if (all_annotations[a] != []): # drop pictures with no trees
                self.annotations.append((a, all_annotations[a]))
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.annotations)  # how many images total

    def __getitem__(self, idx):
        # print(self.annotations)
        # print(self.annotations[1])
        image_name, boxes = self.annotations[idx]
        img = Image.open(self.root_path + "raw_images/" + image_name).convert("RGB")
        img_tensor = self.transform(img)
        
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        
        return img_tensor, target

# Sanity Check:
dataset_full = TreeDataset("./Training Classifier/")
print(f"Total images: {len(dataset_full)}")

Total images: 240


# Step 3
Now we are modifying the pre-trained Faster R-CNN model.  

In particular, we are modifying the classification layer of our Faster R-CNN model. Instead of multiple classes like airplanes, persons, chairs, etc., we will now have **only 2 classes**: tree (1) or no tree (0)

This code is heavily inspired by the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial)

In [3]:
def get_fine_tune_ready_model():
    # load a model pre-trained on COCO
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    num_classes = 2  # 1 class (tree) + background

    # get number of input features for the classifier (from the conv. layers / pooling)
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Predicted trees that overlap more than 30% are the same tree
    model.roi_heads.nms_thresh = 0.3
    
    # Freezing most parameters (everything but the final head/FC layer)
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.rpn.parameters():
        param.requires_grad = False
    
    return model

# Step 4
Now we will finetune the model. This code is almost directly taken from the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial) 

In [4]:
# train on the accelerator (mac) or on the CPU, if an accelerator is not available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# our dataset has two classes only - background and tree
num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model using our helper function
model = get_fine_tune_ready_model()

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.1
)

num_epochs = 24

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

/Users/paulrabel/tree_detector/model/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [ 0/60]  eta: 0:03:11  lr: 0.000090  loss: 5.0791 (5.0791)  loss_classifier: 0.6612 (0.6612)  loss_box_reg: 0.2483 (0.2483)  loss_objectness: 3.8638 (3.8638)  loss_rpn_box_reg: 0.3059 (0.3059)  time: 3.1892  data: 0.0548
Epoch: [0]  [10/60]  eta: 0:01:27  lr: 0.000936  loss: 5.0069 (4.9312)  loss_classifier: 0.5548 (0.5414)  loss_box_reg: 0.3040 (0.2870)  loss_objectness: 3.8338 (3.8895)  loss_rpn_box_reg: 0.2015 (0.2133)  time: 1.7481  data: 0.0738
Epoch: [0]  [20/60]  eta: 0:01:01  lr: 0.001783  loss: 4.4282 (4.5730)  loss_classifier: 0.4248 (0.4472)  loss_box_reg: 0.2641 (0.2639)  loss_objectness: 3.5946 (3.6641)  loss_rpn_box_reg: 0.1912 (0.1979)  time: 1.4535  data: 0.0702
Epoch: [0]  [30/60]  eta: 0:00:44  lr: 0.002629  loss: 3.8517 (4.3194)  loss_classifier: 0.2713 (0.3796)  loss_box_reg: 0.2117 (0.2538)  loss_objectness: 3.1673 (3.4960)  loss_rpn_box_reg: 0.1582 (0.1900)  time: 1.3375  data: 0.0779
Epoch: [0]  [40/60]  eta: 0:00:33  lr: 0.003476  loss: 4.4432 (4.359

# Final Step:
Save the learned weights

In [5]:
torch.save(model.state_dict(), "weights.pt")

# More Adjustments

Currently, we only finetuned the classification layer of our model (the very last layer). Using the following parameters:
- `batch size = 4`
- `learning rate = 0.005`
- `momentum = 0.9`
- `weight decay = 0.0005`
- `step size = 8`
- `gamma = 0.1` (aggressive)
- `epochs = 24`

the classification loss was reduced from $~0.66$ to $~0.10$. 

However, the total loss stayed relatively constant, between $3.5$ and $5$, mainly due to the **loss_objectness** which consistenly is around $3.5$. **loss_objectness** is measuring the performance of our model determining whether any object is at a location. Since the model was trained on `COCO`, and satellite-view trees are not part of that training data, we must also update weights in the RPN layer of the model. Thus, we proceed with *progressive unfreezing*.

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model
model = get_fine_tune_ready_model()
model.load_state_dict(torch.load("weights.pt", weights_only=True))
for param in model.rpn.parameters(): # go deeper than before
    param.requires_grad = True
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=8,
    gamma=0.5
)

num_epochs = 12

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

/Users/paulrabel/tree_detector/model/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [ 0/60]  eta: 0:04:36  lr: 0.000009  loss: 4.5736 (4.5736)  loss_classifier: 0.0783 (0.0783)  loss_box_reg: 0.1005 (0.1005)  loss_objectness: 4.1718 (4.1718)  loss_rpn_box_reg: 0.2230 (0.2230)  time: 4.6005  data: 0.0812
Epoch: [0]  [10/60]  eta: 0:02:16  lr: 0.000094  loss: 3.7864 (3.7183)  loss_classifier: 0.0890 (0.1006)  loss_box_reg: 0.1326 (0.1558)  loss_objectness: 3.2906 (3.2865)  loss_rpn_box_reg: 0.1767 (0.1754)  time: 2.7354  data: 0.0765
Epoch: [0]  [20/60]  eta: 0:01:59  lr: 0.000178  loss: 3.2087 (3.5635)  loss_classifier: 0.1204 (0.1303)  loss_box_reg: 0.1813 (0.1861)  loss_objectness: 2.8007 (3.0724)  loss_rpn_box_reg: 0.1631 (0.1748)  time: 2.8963  data: 0.0962
Epoch: [0]  [30/60]  eta: 0:01:21  lr: 0.000263  loss: 2.8217 (3.1656)  loss_classifier: 0.2625 (0.2330)  loss_box_reg: 0.2883 (0.2654)  loss_objectness: 1.9858 (2.4886)  loss_rpn_box_reg: 0.1676 (0.1787)  time: 2.7274  data: 0.1162
Epoch: [0]  [40/60]  eta: 0:00:53  lr: 0.000348  loss: 2.0658 (2.886

# Final Step

Now that the **loss_objectness** has decreased (from ~$4$ to now being ~$0.25$) and our total **loss** has also significantly decreased, we can save the final weights.

In [8]:
torch.save(model.state_dict(), "final_weights.pt")